# Figures for the SI
## Most (but not all the figures that appear in the SI for the Nature Manuscript)

In [ ]:
%matplotlib inline
import numpy as np
import scipy
import statistics
import matplotlib as mpl
from matplotlib import gridspec
import matplotlib.ticker as ticker
from scipy.optimize import curve_fit
from matplotlib.ticker import (MultipleLocator, AutoMinorLocator, FormatStrFormatter)
from scipy import interpolate
import matplotlib.patches as mpatches
import pandas as pd 
from numpy import *
from scipy.signal import savgol_filter
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.signal import find_peaks
import matplotlib.ticker as plticker
# import probfit
from scipy import special
import time
import datetime
pd.options.mode.copy_on_write = True

mpl.rc('font', family='Arial')

In [ ]:
def readPSDData(filename):
    tmp = pd.read_csv(filename, sep=',', header = None, skiprows=1)
    tmp.columns = ['Energy', 'PSD']
    return tmp

def readFOM(filename):
    tmp = pd.read_csv(filename, sep=',', header = None, skiprows=1)
    tmp.columns = ['i', 'mu1', 'sigma1', 'a1', 'mu2', 'sigma2', 'a2', 'fom', 'e_lower', 'e_higher']
    return tmp

def readNSpectrum(filename):
    tmp = pd.read_csv(filename, sep = ',', header = None, skiprows=2)
    tmp.columns = ['lightOutput(keVee)-sim', 'Sim-NormCounts', 'lightOutput(keVee)-exp', 'Exp-NormCounts']
    return tmp

def _2gaussian(x, amp1,cen1,sigma1, amp2,cen2,sigma2):
    return amp1*(1/(sigma1*(np.sqrt(2*np.pi))))*(np.exp((-1.0/2.0)*(((x_array-cen1)/sigma1)**2))) + \
            amp2*(1/(sigma2*(np.sqrt(2*np.pi))))*(np.exp((-1.0/2.0)*(((x_array-cen2)/sigma2)**2)))

def _1gaussian(x, amp1,cen1,sigma1):
    return amp1*(1/(sigma1*(np.sqrt(2*np.pi))))*(np.exp((-1.0/2.0)*(((x_array-cen1)/sigma1)**2)))

def readNeutronData0(filename):
    tmp = pd.read_csv(filename, sep=',', header = None, skiprows=0)
    return tmp

In [ ]:
def readNeutronData(filename):
    tmp = pd.read_csv(filename, sep=',', header = None, skiprows=1)
    tmp.columns = ['binMid', 'binTime(s)', 'Neutrons(cps)','delta-neutrons', 'BackgroundGamma(cps)', 'delta-gamma']
    tmp['binTime(m)'] = tmp['binTime(s)']/60
    return tmp

def readVapor(filename):
    tmp = pd.read_csv(filename, sep=',', header = None, skiprows=0)
    return tmp

In [ ]:
data_folder = "C:/Users/oliver.horner/Desktop/Nature paper raw data/Output/"
id419_5sigma = readNeutronData(data_folder + 'ID-419/ID-419_data_300s_bin.csv')
id423_5sigma = readNeutronData(data_folder + 'ID-423/ID-423_data_300s_bin.csv')

# id419_4sigma = readNeutronData(data_folder + 'ID-419/ID-419_data_60s_bin-4sigma.csv')
# id423_4sigma = readNeutronData(data_folder + 'ID-423/ID-423_data_60s_bin-4sigma.csv')

# id419_3sigma = readNeutronData(data_folder + 'ID-419/ID-419_data_300s_bin-3sigma.csv')
# id423_3sigma = readNeutronData(data_folder + 'ID-423/ID-423_data_300s_bin-3sigma.csv')

--------

In [ ]:
psdData = readPSDData('Archive/psd_data.csv')
FOM = readFOM('Archive/fom_analysis.csv')
data = readVapor('../DataToPlot/VaporWaveIsland.csv') # this is the vapour wave island that was used to generate the 3D plot

In [ ]:
data.replace(0, np.nan, inplace=True)

In [ ]:
from mpl_toolkits.mplot3d import Axes3D


x = data.columns
y = data.index
X,Y = np.meshgrid(x,y)
Z = data

ax.grid(False)

fig = plt.figure(figsize = (8,8))
ax = fig.add_subplot(111, projection='3d')
ax.view_init(40,50)
ax.plot_surface(-X/100, Y*15, Z, color = '#909090ff')

# Get rid of colored axes planes
# First remove fill
ax.xaxis.pane.fill = False
ax.yaxis.pane.fill = False
ax.zaxis.pane.fill = False

# Now set color to white (or whatever is "invisible")
ax.xaxis.pane.set_edgecolor('w')
ax.yaxis.pane.set_edgecolor('w')
ax.zaxis.pane.set_edgecolor('w')


ax.set_xlabel('Pulse shape discrimination (arb. unit)')
ax.set_ylabel('Light Output (keVee)')
ax.set_zlabel('Counts')

# Bonus: To get rid of the grid as well:
plt.savefig("Fig2a-Contour.pdf", format="pdf")


plt.show()
# The red transparency that appears in the text was placed using Inkscape and drawing out the boundaries

------

# SI Figure 4b - 2D PSD plot

In [ ]:
fig, axs = plt.subplots(1, 1, figsize = (10,7))
fig.tight_layout(pad=2)
threshold = 50
otherThreshold = 640
axs.plot(xaxis[16:], yaxis[16:],  ls ='-', color = 'red', lw = 1)
axs.plot(xaxis[16:], yaxis[16:]+0.2,  ls ='-', color = 'red', lw = 1)
axs.fill_between(xaxis, yaxis,yaxis+0.2, where=xaxis>threshold, color = 'red', alpha = 0.2)
# axs.plot(psdData['Energy'], psdData['PSD'], ls ='', marker = 'o', ms = 2, alpha = 0.1, color = 'black')
histresult = axs.hist2d(psdData['Energy'], psdData['PSD'], bins = 300, cmin=2, cmap = 'viridis')
img = histresult[3]

axs.set_xlim(0,1650)
axs.set_ylim(0,0.5)
axs.set_ylabel('Pulse shape discrimination (arb. unit)', fontsize=24)

axs.annotate('Gamma-ray channel',(300,0.07), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 22, fontname = 'Arial',color = 'black' )
axs.set_xlabel('Light output (keVee)', fontsize=24)
# axs[0].axhline(0.35, color = 'black', ls = "--", alpha = 0.7)
# axs.axhline(0.16, color = 'black', ls = "--", alpha = 0.7)
# axs.axvline(195,color = 'black', ls ='--')
axs.axvline(195,color = '#ffcb00ff', ls ='-')
# axs[0].axvline(100, color = 'black')


axs.tick_params(axis="x", labelsize = 22)
axs.tick_params(axis="y", labelsize = 22)
cbar = fig.colorbar(img, ax = axs)
cbar.set_label('Counts', fontsize = 24, rotation = 270, labelpad = 15)
cbar.ax.tick_params(labelsize = 22)

# axs.axvline(195, color = 'red')
# axs.axvline(205, color = 'red')

axs.annotate('Neutron channel',(300,0.43), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 22, fontname = 'Arial', color = 'black' )
plt.savefig("SI-Fig4a-PSDPlot.pdf", format="pdf", bbox_inches="tight")

plt.show()


# SI Figure 4a - Waveform example

In [ ]:
def readwaveforms(filename):
    tmp = pd.read_csv(filename, sep=',', header = None, skiprows=1)
    tmp.columns = ['index', 'neutron', 'gamma']
    return tmp


waveforms = readwaveforms('../DataToPlot/ID423/ID-423_signal_comparison.csv')

In [ ]:
fig, axs = plt.subplots(1, 1, figsize = (7,7))
fig.tight_layout(pad=2)

# axs.errorbar(id325og['binTime(s)'], id325og['Neutrons(cps)'], id325og['delta-neutrons']*1.5, ls ='', marker = 's', ms = 3, capsize = 3, label = 'ID-325', color ='#4574A2FF' )
axs.plot(waveforms['index'], waveforms['neutron'], ls ='-', lw = 3, color = 'C0', label = 'neutrons')
axs.plot(waveforms['index'], waveforms['gamma'], ls ='--', lw = 3, color = 'C1', label = 'neutrons')
# axs.errorbar(id378['binTime(s)'], id378['scaledCounts'], id378['delta-neutrons'], ls ='', marker = 's', ms = 3, capsize = 3, label = 'ID-375 - Target A4', color ='#424242FF' )
# axs.set_title('Beam Loading', fontsize = 16)
# axs.legend(fontsize = 14, loc = 'lower right')
axs.set_xlim(-7.5,220)
axs.set_ylim(-650,2100)
axs.set_xlabel('Time (arb. units)', fontsize=24)
axs.set_ylabel('Pulse height (arb. units)', fontsize=24)



fs = 13.0
[t.set_fontsize(fs) for t in axs.xaxis.get_majorticklabels()]
[t.set_fontsize(fs) for t in axs.yaxis.get_majorticklabels()]
axs.yaxis.get_offset_text().set_fontsize(fs)

# Here is the label and arrow code of interest
axs.annotate('Tail gate', xy=(138,-200), xytext=(138, -250), xycoords='data', 
            fontsize=fs*1.5, ha='center', va='bottom',
            bbox=dict(boxstyle='square', fc='white', color='k'),
            arrowprops=dict(arrowstyle='-[, widthB=6.5, lengthB=-0.5', lw=2.0, color='k'))

axs.annotate('Total gate', xy=(120,-400), xytext=(120, -450), xycoords='data', 
            fontsize=fs*1.5, ha='center', va='bottom',
            bbox=dict(boxstyle='square', fc='white', color='k'),
            arrowprops=dict(arrowstyle='-[, widthB=8.3, lengthB=-0.5', lw=2.0, color='k'))
axs.tick_params(axis="x", labelsize = 22)
axs.tick_params(axis="y", labelsize = 22)


plt.savefig("WaveformComparison.png", format="png", bbox_inches="tight")

plt.show()



# SI Figure 5 -  Energy calibration plot 

In [ ]:
def readCalData(filename):
    tmp = pd.read_csv(filename, sep=',', header = None, skiprows=1)
    tmp.columns = ['Isotope','EnergyPeak', 'LightOutput', 'ADCChannel']
    return tmp

In [ ]:
calData = readCalData('CalibrationCurveData.csv')

In [ ]:
csData = calData.iloc[0]
coData = calData.iloc[1:3]
euData = calData.iloc[3:8]

In [ ]:
x = np.linspace(0, 2600, 100)

In [ ]:
fig, axs = plt.subplots(1, 1, figsize = (7,7))
fig.tight_layout(pad=2)

# axs.errorbar(id325og['binTime(s)'], id325og['Neutrons(cps)'], id325og['delta-neutrons']*1.5, ls ='', marker = 's', ms = 3, capsize = 3, label = 'ID-325', color ='#4574A2FF' )
axs.plot(coData['ADCChannel'], coData['LightOutput'], ls ='', marker = 'o', ms = 8, color = 'blue', label = '$^{60}$Co')
axs.plot(csData['ADCChannel'], csData['LightOutput'], ls ='', marker = 's', ms = 6, color = 'black', label = '$^{137}$Cs')
axs.plot(euData['ADCChannel'], euData['LightOutput'], ls ='', marker = 'p', ms = 8, color = 'red', label = '$^{152}$Eu')
axs.plot(x, (x-41.67)/2.179, linestyle='--', alpha = 0.5, label = 'Fit') 
# axs.errorbar(id378['binTime(s)'], id378['scaledCounts'], id378['delta-neutrons'], ls ='', marker = 's', ms = 3, capsize = 3, label = 'ID-375 - Target A4', color ='#424242FF' )
# axs.set_title('Beam Loading', fontsize = 16)
axs.legend(fontsize = 14, loc = 'lower right')
# axs.set_xlim(-33,1700)
# axs.set_ylim(-0.04,0.7)
axs.set_xlabel('Analog-to-digital converter channel (arb. units)', fontsize=16)
axs.set_ylabel('Light output (keVee)', fontsize=16)

axs.tick_params(axis="x", labelsize = 14)
axs.tick_params(axis="y", labelsize = 14)
# axs.axvline(195, color = 'red')
# axs.axvline(205, color = 'red')
# axs.plot(xaxis, yaxis,  ls ='--', color = 'C1')
# axs.plot(xaxis, yaxis+0.2,  ls ='--', color = 'C1')
# axs.fill_between(xaxis, yaxis,yaxis+0.2, where=xaxis>threshold, color = 'C1', alpha = 0.3)

plt.savefig("GammaCalibration.pdf", format="pdf", bbox_inches="tight")

plt.show()



# Fig 2b in the manuscript and SI Fig 4c

In [ ]:
data = readNeutronData0('VaporWaveIsland.csv') # Sorry for the renaming functions and data... it got dicy...

In [ ]:
neutron_sigmaValues = []
gamma_sigmaValues = []
neutron_centroids = []
gamma_centroids = []
energy_Slices = []

for i in range(13,14):
    amp1 = 100
    sigma1 = 8
    cen1 = 25
    amp2 = 200
    sigma2 = 10
    cen2 = 72
    
    x_array = data.iloc[i].index
    y_array_2gauss = data.iloc[i].values
    
    popt_2gauss, pcov_2gauss = scipy.optimize.curve_fit(_2gaussian, x_array, y_array_2gauss, p0=[amp1, cen1, sigma1, amp2, cen2, sigma2])
    perr_2gauss = np.sqrt(np.diag(pcov_2gauss))
    pars_1 = popt_2gauss[0:3]
    pars_2 = popt_2gauss[3:6]
    gauss_peak_1 = _1gaussian(x_array, *pars_1)
    gauss_peak_2 = _1gaussian(x_array, *pars_2)

    FOM = abs(popt_2gauss[4] - popt_2gauss[1])/((2.36*(popt_2gauss[2]+popt_2gauss[5])))

    neutron_sigmaValues.append(popt_2gauss[5])
    gamma_sigmaValues.append(popt_2gauss[2])
    neutron_centroids.append(popt_2gauss[4])
    gamma_centroids.append(popt_2gauss[1])
    energy_Slices.append(i*15)
    
    fig, ax = plt.subplots(1, 1, figsize = (7,7))
    fig.tight_layout()
    # ax.set_title('Slice #' + str(i) +' - '+ str(i*15) + ' keVee', fontsize = 16)
    # ax.plot(x_array*0.005, _2gaussian(x_array, *popt_2gauss), color = 'black')#,\
    ax.plot(data.iloc[i].index*0.005,data.iloc[i].values, color = 'black', lw=3)
    ax.plot(x_array*0.005, gauss_peak_1, color="#848484FF", lw=3) ## gamma
    ax.set_xlabel('Pulse shape discriminarion (arb. unit)', fontsize = 24)
    ax.set_ylabel('Counts', fontsize = 24)
    ax.fill_between(x_array*0.005, gauss_peak_1.min(), gauss_peak_1, facecolor="#424242FF", alpha=0.2)
    ax.plot(x_array*0.005, gauss_peak_2, color="#941100FF", lw=3) ## neutron
    ax.fill_between(x_array*0.005, gauss_peak_2.min(), gauss_peak_2, facecolor="#941100FF", alpha=0.2)
    ax.tick_params(axis="x", labelsize = 24)
    ax.tick_params(axis="y", labelsize = 24)
    # ax.annotate('FOM = ' + (str(FOM)),(0,10), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 14 )
    ax.annotate(r'$\mathrm{\gamma}$-rays',(0.05,100), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 24, ha='center', color="#424242FF")
    ax.annotate('Neutrons',(0.31,140), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 24, ha="right", color="#941100FF")
    ax.annotate(r"$\mathrm{\mu_{\gamma}}$", (popt_2gauss[1]*0.005 - 0.01, 270), xytext=None, xycoords='data', textcoords='data', fontsize=24, ha='right', va='center', color='black')
    ax.annotate(r"$\mathrm{5\sigma}$", (popt_2gauss[1]*0.005 + popt_2gauss[2]*5*0.005 - 0.01, 270), xytext=None, xycoords='data', textcoords='data', fontsize=24, ha='right', va='center', color='#941100FF')
    ax.annotate(r"$\mathrm{\mu_{n}}$", (popt_2gauss[4]*0.005 + 0.01, 270), xytext=None, xycoords='data', textcoords='data', fontsize=24, ha='left', va='center', color='black')
    # ax.annotate('3$\sigma$',(1+popt_2gauss[4]-popt_2gauss[5]*3,250), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 14, rotation = 90 )
    ax.set_ylim(None, 285)
    
# plot slices into the same graph
# for i in range(10,11):
#     plt.plot(data.iloc[i].index,data.iloc[i].values, alpha = 0.4)

    plt.axvline(popt_2gauss[1]*0.005, lw=3, color = 'black')
    plt.axvline(popt_2gauss[1]*0.005 + popt_2gauss[2]*5*0.005, ls = '--', lw=3, color = '#941100FF')
    plt.axvline(popt_2gauss[4]*0.005, lw=3, color = 'black')

    plt.savefig("Fig2b-PSDSlice.png", format="png", bbox_inches="tight") 
    
    plt.show()